# Convergence Plots

Plots average best-so-far error vs. function evaluations for each function.

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("D:/Developments/Optimization_data/RCMAES/bin")
ALGORITHMS = ['arrde', 'nlshade_rsp', 'rcmaes', 'jso', 'lsrtde', "bipop_acmaes", "j2020"]
Algo_proper_name = {
    'arrde': 'ARRDE',
    'nlshade_rsp': 'NL-SHADE-RSP',
    'rcmaes': 'RCMAES',
    'jso': 'jSO',
    'lsrtde': 'LSRTDE',
    'bipop_acmaes': 'BIPOP-aCMAES', 
    "j2020": "J2020"
}
ALGOS_UP = [a.upper() for a in ALGORITHMS]

def load_algo_files(algo: str):
    algo_dir = BASE_DIR / algo
    if not algo_dir.exists():
        print(f'Warning: {algo_dir} not found')
        return {}
    files = list(algo_dir.glob(f'{algo}_F*_Min_EV.txt'))
    data = {}
    for f in files:
        m = re.search(r'_F(\d+)_Min_EV', f.name)
        if not m:
            continue
        func = int(m.group(1))
        mat = np.loadtxt(f, delimiter='	')
        if mat.ndim == 1:
            mat = mat.reshape(-1, 1)
        data[func] = mat
    return data

algo_data = {algo: load_algo_files(algo) for algo in ALGORITHMS}

all_funcs = sorted({fn for data in algo_data.values() for fn in data.keys()})
print('Functions found:', all_funcs)

def eval_axis(num_points, dimension):
    step = max(1, 10 * dimension)
    return np.arange(num_points) * step

def infer_dimension(num_points, max_evals):
    if num_points <= 1:
        return 1
    step = max_evals / (num_points - 1)
    return max(1, int(round(step / 10)))


In [ ]:
rows, cols = 6, 5
figsize = (15, 15)

# If you know max_evals and dimension, set them here for correct x-axis:
MAX_EVALS = None  # e.g., 10000 * dimension
DIMENSION = None
chunk = all_funcs
fig, axes = plt.subplots(rows, cols, figsize=figsize, sharex=False, sharey=False)
axes = axes.flatten()
for ax in axes:
    ax.axis('off')

for ax_idx, func in enumerate(chunk):
    if ax_idx ==1 : 
        continue
    if ax_idx>1 : ax_idx = ax_idx-1
    func_label = func 
    if func_label>2 : 
        func_label = func_label - 1
    ax = axes[ax_idx]
    for algo, algo_up in zip(ALGORITHMS, ALGOS_UP):
        data = algo_data.get(algo, {}).get(func)
        if data is None:
            continue
        ax.axis('on')
        data = np.where(np.isfinite(data), data, np.nan)
        y = np.nanmean(data, axis=1)
        fstar = func * 100
        y = y - fstar
        #y = np.where(y <= 0, 1e-7, y)
        if DIMENSION is not None and MAX_EVALS is not None:
            x = eval_axis(len(y), DIMENSION)
        else:
            if MAX_EVALS is not None:
                dim = infer_dimension(len(y), MAX_EVALS)
                x = eval_axis(len(y), dim)
            else:
                x = np.arange(len(y))
        ax.plot(x, y, label=algo_up)
    ax.set_yscale('log')
    ax.text(0.1, 0.95, f'F{func_label}', transform=ax.transAxes, fontsize=9, va='top')
    ax.grid(True, which='both', alpha=0.3)
    ax.set_yscale("log")
    #ax.set_ylim(1e-10)
    #ax.tick_params(axis='y', direction='in', pad=-8)
    
for ax in axes[len(chunk):]:
    ax.axis('off')
handles, labels = axes[0].get_legend_handles_labels()
labels = [Algo_proper_name.get(label.lower(), label) for label in labels]
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=len(ALGORITHMS), bbox_to_anchor=(0.5, 0.96))
    fig.subplots_adjust(top=0.5)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.subplots_adjust(wspace=0.2)
plt.savefig("curve.pdf", bbox_inches='tight')
